# Blood-Based Biomarkers for Tissue Pathology (GTEx)

This notebook builds pipeline to test whether **Whole Blood gene expression** can predict **tissue pathology labels**
(example: **liver steatosis**).



## 1. Load Required Libraries

In [1]:
# Imports
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from collections import Counter

import sklearn
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_curve, roc_auc_score, mean_absolute_error,
    precision_recall_curve, average_precision_score,
    auc as sk_auc,
    confusion_matrix, ConfusionMatrixDisplay,
)

# Reproducibility
np.random.seed(0)

# Version check
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sklearn:", sklearn.__version__)


pandas: 2.2.2
numpy: 1.26.4
sklearn: 1.7.1


## 2. Paths and Directories

In [2]:
# Paths and directories
root_dir = Path("..")
raw_dir = root_dir / "data" / "raw"
processed_dir = root_dir / "data" / "processed"

tables_dir = root_dir / "output" / "tables"
figures_dir = root_dir / "output" / "figures"

for folder in [raw_dir, processed_dir, tables_dir, figures_dir]:
    folder.mkdir(parents=True, exist_ok=True)

raw_dir, tables_dir, figures_dir


(PosixPath('../data/raw'),
 PosixPath('../output/tables'),
 PosixPath('../output/figures'))

## 3. Load Input Files

- **GTEx gene TPM matrix** — expression across all tissues
- **GTEx sample attributes** — sample-level metadata
- **GTEx restricted subject table** — donor AGE
- **Pathology / metadata table** — tissue pathology categories + notes

In [ ]:
# Input files
file_path_expr = raw_dir / "GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct"
file_path_meta = raw_dir / "GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"
file_path_age  = raw_dir / "Gtex_restricted.txt"
file_path_pathology = raw_dir / "meta_data_with_url.csv"

# Expression 
df_expr = pd.read_csv(file_path_expr, sep="\t", skiprows=2)

# Sample attributes
df_samples = pd.read_csv(file_path_meta, sep="\t", low_memory=False)

# Restricted Data for Age
df_age = pd.read_csv(file_path_age, sep="\t", low_memory=False)

# Pathology  metadata for tissue pathology
df_meta_url = pd.read_csv(file_path_pathology, low_memory=False)

print("Expression shape:", df_expr.shape)
print("Sample attributes shape:", df_samples.shape)
print("Restricted (age) shape:", df_age.shape)
print("Meta/pathology shape:", df_meta_url.shape)


## 4. Explore Loaded DataFrames

In [ ]:
df_expr.head()

In [ ]:
df_samples 


In [ ]:
df_age 

In [ ]:
df_meta_url

## 5. Filter to Whole Blood Samples

`SMTS == 'Blood'`, keep `SMTSD == 'Whole Blood'` and remove EBV-transformed lymphocytes

In [ ]:
# Filter metadata to Blood
blood_meta = df_samples[df_samples["SMTS"] == "Blood"].copy()

# Remove EBV-transformed lymphocytes
blood_meta = blood_meta[blood_meta["SMTSD"] != "Cells - EBV-transformed lymphocytes"].copy()

# Keep Whole Blood
blood_meta = blood_meta[blood_meta["SMTSD"] == "Whole Blood"].copy()

print("Whole Blood samples in metadata:", blood_meta["SAMPID"].nunique())
blood_meta[["SAMPID", "SMTS", "SMTSD"]]


## 6. Match Whole Blood Samples to Expression Matrix

The expression table stores genes in rows and samples in columns — we transpose
to get a **(samples × genes)** matrix.

In [ ]:
# Match Whole Blood sample IDs between metadata and expression table
expr_sample_cols = list(df_expr.columns[2:])  # sample IDs start after Name/Description
whole_blood_ids = set(blood_meta["SAMPID"].astype(str))

overlap_wb = [sid for sid in expr_sample_cols if sid in whole_blood_ids]
print("Whole Blood samples matched in expression:", len(overlap_wb))

# Expression subset: keep Name, Description + Whole Blood samples
df_blood_wb = df_expr[["Name", "Description"] + overlap_wb].copy()
print("df_blood_wb shape:", df_blood_wb.shape)

# Build matrix: rows=samples, cols=genes
expr_numeric = df_blood_wb[overlap_wb].copy()
expr_numeric.index = df_blood_wb["Name"].astype(str)  # gene IDs

X_wb = expr_numeric.T  # samples x genes
X_wb = X_wb.apply(pd.to_numeric, errors="coerce").fillna(0)

print("X_wb shape (samples x genes):", X_wb.shape)
print("Example sample IDs:", X_wb.index[:3].tolist())
print("Example gene IDs:", X_wb.columns[:3].tolist())


## 7. PCA on Whole Blood Expression

In [ ]:
# PCA on scaled expression
X_scaled = StandardScaler().fit_transform(X_wb)
pca = PCA(n_components=10, random_state=0)
pcs = pca.fit_transform(X_scaled)

print("Explained variance (PC1-10):", pca.explained_variance_ratio_)


### 7.1 Cumulative Explained Variance

In [ ]:
var = pca.explained_variance_ratio_
cum_var = np.cumsum(var)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker="o")
plt.title("Cumulative Explained Variance (Whole Blood PCA)")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.grid(True)
plt.savefig(figures_dir / "pca_whole_blood_cumulative_explained_variance.png", dpi=300, bbox_inches="tight")
plt.show()


### 7.2 PC1 vs PC2 — colored by SEX & AGE

SEX from pathology metadata, AGE from df_age. Both are donor-level → merge via **SUBJID** derived from the sample ID.

In [ ]:
# PCA scores table
pc_cols = [f"PC{i}" for i in range(1, pcs.shape[1] + 1)]
pca_scores = pd.DataFrame(pcs, index=X_wb.index, columns=pc_cols).reset_index()
pca_scores = pca_scores.rename(columns={"index": "SAMPID"})

# Donor ID (SUBJID) from SAMPID like: GTEX-XXXX-...
pca_scores["SUBJID"] = pca_scores["SAMPID"].astype(str).str.split("-").str[:2].str.join("-")

# SEX from df_meta_url
# Make sure SUBJID exists
if "SUBJID" not in df_meta_url.columns:
    # common pattern: Tissue.Sample.ID exists
    if "Tissue.Sample.ID" in df_meta_url.columns:
        df_meta_url["SUBJID"] = df_meta_url["Tissue.Sample.ID"].astype(str).str.split("-").str[:2].str.join("-")

# Find a sex column (prefer exact 'SEX')
sex_col = "SEX" if "SEX" in df_meta_url.columns else None
if sex_col is None:
    sex_candidates = [c for c in df_meta_url.columns if "sex" in c.lower() or "gender" in c.lower()]
    if len(sex_candidates) > 0:
        sex_col = sex_candidates[0]
        print("Using SEX column:", sex_col)
    else:
        print("No SEX column found in meta_data_with_url.csv; continuing without SEX.")

if sex_col is not None and "SUBJID" in df_meta_url.columns:
    sex_df = (
        df_meta_url[["SUBJID", sex_col]]
        .dropna()
        .drop_duplicates("SUBJID")
        .rename(columns={sex_col: "SEX"})
    )
else:
    sex_df = pd.DataFrame(columns=["SUBJID", "SEX"])



In [ ]:
# AGE from df_age
# Ensure donor id column is named SUBJID
if "SUBJID" not in df_age.columns:
    # Try common alternatives
    for c in df_age.columns:
        if c.lower() in ["subject_id", "subjectid", "subjid", "gtex_donor_id"]:
            df_age = df_age.rename(columns={c: "SUBJID"})
            break

# Choose AGE column
if "AGE" in df_age.columns:
    age_col = "AGE"
else:
    age_candidates = [c for c in df_age.columns if "age" in c.lower()]
    if len(age_candidates) == 0:
        raise KeyError("No AGE-like column found in Gtex_restricted.txt. Inspect df_age.columns and pick the correct one.")
    age_col = age_candidates[0]
    print("Using AGE column:", age_col)

age_df = (
    df_age[["SUBJID", age_col]]
    .dropna()
    .drop_duplicates("SUBJID")
    .rename(columns={age_col: "AGE"})
)

# Merge for plotting
plot_df = (
    pca_scores
    .merge(blood_meta[["SAMPID", "SMTSD"]], on="SAMPID", how="left")
    .merge(sex_df, on="SUBJID", how="left")
    .merge(age_df, on="SUBJID", how="left")
)

In [ ]:
# PCA colored by SEX 
if plot_df["SEX"].notna().any():
    plt.figure(figsize=(9, 6))
    for s, sub in plot_df.groupby("SEX"):
        plt.scatter(sub["PC1"], sub["PC2"], alpha=0.5, s=10, label=f"SEX={s}")
    plt.title("PCA of Whole Blood Expression (PC1 vs PC2) colored by SEX")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(True)
    plt.legend()
    plt.savefig(figures_dir / "pca_whole_blood_pc1_pc2_by_sex.png", dpi=300, bbox_inches="tight")
    plt.show()

plot_df[["SAMPID", "SUBJID", "PC1", "PC2", "SEX", "AGE"]].head()


## 8. Create Liver Steatosis Label from Pathology Metadata

A donor is labeled steatosis-positive if **'steatosis'** appears in pathology
fields for **Liver**. This is a simple text-based label.

In [ ]:
# Identify required columns in df_meta_url
tissue_col = "Tissue" if "Tissue" in df_meta_url.columns else None
samp_col = "Tissue.Sample.ID" if "Tissue.Sample.ID" in df_meta_url.columns else None
cat_col = "Pathology.Categories" if "Pathology.Categories" in df_meta_url.columns else None
notes_col = "Pathology.Notes" if "Pathology.Notes" in df_meta_url.columns else None

missing = [n for n, c in [("Tissue", tissue_col), ("Tissue.Sample.ID", samp_col), ("Pathology.Categories", cat_col)] if c is None]
if len(missing) > 0:
    raise KeyError(f"meta_data_with_url.csv is missing required columns: {missing}")

liver_path = df_meta_url.loc[df_meta_url[tissue_col].astype(str) == "Liver", [samp_col, tissue_col, cat_col] + ([notes_col] if notes_col else [])].copy()
liver_path = liver_path.rename(columns={samp_col: "SAMPID"})

# Combine text fields for robust matching
text = liver_path[cat_col].fillna("").astype(str)
if notes_col:
    text = text + " " + liver_path[notes_col].fillna("").astype(str)

liver_path["steatosis_label"] = text.str.lower().str.contains("steatosis").astype(int)

# Donor ID
liver_path["SUBJID"] = liver_path["SAMPID"].astype(str).str.split("-").str[:2].str.join("-")

# Donor-level label: positive if ANY liver sample is positive
donor_steatosis = liver_path.groupby("SUBJID")["steatosis_label"].max()

print("Liver samples with steatosis_label=1:", int(liver_path["steatosis_label"].sum()))
print("Donors with steatosis_label=1:", int(donor_steatosis.sum()))


## 9. Build Labeled Blood Dataset (X_features, y, groups)

Map the donor-level liver label onto Whole Blood samples from the same donors.

In [ ]:
# Map each blood sample to donor
blood_subjid = pd.Series(X_wb.index, index=X_wb.index).astype(str).str.split("-").str[:2].str.join("-")

# Map donor label to each blood sample
y = blood_subjid.map(donor_steatosis)

# Keep only samples with known label
keep = y.notna()
X_features = X_wb.loc[keep].copy()
y = y.loc[keep].astype(int)

# groups (donor IDs) for grouped CV
groups = blood_subjid.loc[keep].astype(str)

print("Labeled Whole Blood samples:", X_features.shape[0])
print("Features (genes):", X_features.shape[1])
print("Positive samples:", int(y.sum()), " / ", int(y.shape[0]))

# Save labels table for audit
labels_df = pd.DataFrame({
    "SAMPID": X_features.index.astype(str),
    "SUBJID": groups.values,
    "steatosis_label": y.values
})
labels_df.to_csv(tables_dir / "labels_blood_samples_liver_steatosis.csv", index=False)
labels_df.head()


## 10. Visualize Label Skewness

In [ ]:
# Sample-level label distribution
sample_counts = y.value_counts().sort_index()
sample_pct = (sample_counts / sample_counts.sum() * 100).round(2)

plt.figure(figsize=(6, 10))
plt.bar(sample_counts.index.astype(str), sample_counts.values)
plt.title("Steatosis label distribution (samples)")
plt.xlabel("steatosis_label")
plt.ylabel("Number of blood samples")
for i, v in enumerate(sample_counts.values):
    plt.text(i, v, f"{v}\n({sample_pct.iloc[i]}%)", ha="center", va="bottom")
plt.savefig(figures_dir / "label_distribution_samples.png", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
# Donor-level label distribution
donor_counts = donor_steatosis.value_counts().sort_index()
donor_pct = (donor_counts / donor_counts.sum() * 100).round(2)

plt.figure(figsize=(6, 10))
plt.bar(donor_counts.index.astype(str), donor_counts.values)
plt.title("Steatosis label distribution (donors)")
plt.xlabel("donor steatosis_label")
plt.ylabel("Number of donors")
for i, v in enumerate(donor_counts.values):
    plt.text(i, v, f"{v}\n({donor_pct.iloc[i]}%)", ha="center", va="bottom")
plt.savefig(figures_dir / "label_distribution_donors.png", dpi=300, bbox_inches="tight")
plt.show()


## 11. PCA Colored by Steatosis Label

In [ ]:
# Merge labels onto PCA points
pca_label_df = pca_scores.merge(labels_df.rename(columns={"SAMPID": "SAMPID"}), on="SAMPID", how="inner")

plt.figure(figsize=(9, 6))
for lab, sub in pca_label_df.groupby("steatosis_label"):
    plt.scatter(sub["PC1"], sub["PC2"], alpha=0.6, s=10, label=f"label={lab}")
plt.title("Whole Blood PCA (PC1 vs PC2) colored by liver steatosis label")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.legend()
plt.savefig(figures_dir / "pca_whole_blood_pc1_pc2_by_steatosis.png", dpi=300, bbox_inches="tight")
plt.show()


## 12. 5-Fold Stratified Cross-Validation with Feature Selection
For each fold:
1. Use 4/5 folds (training) to score features (per-gene AUC strength).
2. Select **top 100** features.
3. Train LogisticRegression on training fold.
4. Test on held-out fold.
Feature selection repeats **5 times** (once per fold) — So there no data leakage.

In [ ]:
# Settings

selection_method = "auc"   # "corr" (fast) or "auc" (slow)
k_features = 100

# Model (simple baseline)
model_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(solver="saga", max_iter=5000))
])

# Grouped CV to avoid donor leakage
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

fold_rows = []
selected_features_per_fold = []
fprs = [] 
tprs = [] 
aucs = []

# Store out-of-fold predictions for later plots (ROC / PR / confusion matrix / boxplot) 
oof_proba = np.empty(len(y)) 
oof_proba[:] = np.nan

for fold, (train_idx, test_idx) in enumerate(cv.split(X_features, y, groups=groups), start=1):
    X_train = X_features.iloc[train_idx]
    X_test  = X_features.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    # ---- Feature selection (TRAIN ONLY) ----
    if selection_method == "corr":
        scores = X_train.corrwith(y_train).fillna(0).abs()

    elif selection_method == "auc":
        # Per-feature AUC strength on training set: |AUC - 0.5|
        strengths = {}
        for col in X_train.columns:
            s = pd.to_numeric(X_train[col], errors="coerce").fillna(0)
            if s.nunique() < 2:
                strengths[col] = 0.0
                continue
            a = roc_auc_score(y_train, s)
            strengths[col] = abs(a - 0.5)
        scores = pd.Series(strengths).reindex(X_train.columns).fillna(0)

    else:
        raise ValueError("selection_method must be 'corr' or 'auc'")

    top_features = scores.nlargest(k_features).index.tolist()
    selected_features_per_fold.append(top_features)

    # ---- Train + test ----
    model_pipe.fit(X_train[top_features], y_train)
    proba = model_pipe.predict_proba(X_test[top_features])[:, 1]
    # Save out-of-fold probabilities for the test samples in this fold
    oof_proba[test_idx] = proba     
    fold_auc = roc_auc_score(y_test, proba)

    fpr, tpr, _ = roc_curve(y_test, proba) 
    fprs.append(fpr) 
    tprs.append(tpr) 
    aucs.append(fold_auc)

    # MAE on hard predictions (0/1) - optional
    y_pred = (proba >= 0.5).astype(int)
    mae = mean_absolute_error(y_test, y_pred)

    fold_rows.append({"fold": fold, "auc": fold_auc, "mae": mae})
    print(f"Fold {fold}: AUC={fold_auc:.3f}, MAE={mae:.3f}")

results_df = pd.DataFrame(fold_rows)
results_df.to_csv(tables_dir / "cv_results_leakfree_top100.csv", index=False)

print("\nSummary:")
display(results_df)
print("Mean AUC:", float(results_df["auc"].mean()))
print("Std AUC:", float(results_df["auc"].std()))

## 13. Fold Performance Plot

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(results_df["fold"], results_df["auc"], marker="o")
plt.title("AUC per fold (top-100 selected inside fold)")
plt.xlabel("Fold")
plt.ylabel("AUC")
plt.ylim(0, 1)
plt.grid(True)
plt.savefig(figures_dir / "cv_auc_per_fold.png", dpi=300, bbox_inches="tight")
plt.show()


## 14. Feature Stability Across Folds
We annotate Ensembl IDs with readable gene symbols from the expression file.

In [ ]:
# Count how many folds each feature was selected in
counts = Counter([g for fold_list in selected_features_per_fold for g in fold_list])
stable = pd.Series(counts).sort_values(ascending=False)

stable_df = stable.reset_index()
stable_df.columns = ["gene", "times_selected"]
stable_df.to_csv(tables_dir / "feature_stability_counts.csv", index=False)

print("Top 20 repeatedly selected features:")
display(stable_df.head(20))

# Build a lookup table from df_expr (Ensembl ID → gene symbol)
gene_info = (
    df_expr[["Name", "Description"]]
    .drop_duplicates(subset=["Name"])
    .rename(columns={"Name": "gene", "Description": "gene_symbol"})
)

stable_annot = stable_df.merge(gene_info, on="gene", how="left")
print(f"Missing gene symbols: {stable_annot['gene_symbol'].isna().sum()}")

# Plot top 20 with readable gene labels
top20 = stable_annot.head(20).iloc[::-1].copy()
top20["label"] = top20["gene_symbol"].fillna(top20["gene"])

plt.figure(figsize=(10, 6))
plt.barh(top20["label"].astype(str), top20["times_selected"])
plt.title("Most repeatedly selected genes (top 20)")
plt.xlabel("# folds selected in (out of 5)")
plt.ylabel("Gene")
plt.tight_layout()
plt.savefig(figures_dir / "top20_stable_features_with_symbols.png", dpi=300, bbox_inches="tight")
plt.show()


## 15. Evaluation Plots (Binary Steatosis Model)

### 15.1 Box Plot — Predicted Probability by Ground-Truth Label

In [ ]:
# Box plot: x = predicted probability, y = ground truth label (0/1) (out-of-fold predictions only)

p0 = oof_proba[y.values == 0]
p1 = oof_proba[y.values == 1]

plt.figure(figsize=(8, 4))
plt.boxplot([p0[~np.isnan(p0)], p1[~np.isnan(p1)]], vert=False, tick_labels=["0", "1"])

plt.axvline(0.5, color="red", linestyle="--", linewidth=2)  # threshold line

ax = plt.gca()
ticks = ax.get_xticks()
ax.set_xticks(np.sort(np.unique(np.append(ticks, 0.5))))     

plt.xlim(0, 1)
plt.xlabel("Predicted probability (class=1)")
plt.ylabel("Ground truth label")
plt.title("Cross-validated predicted probability by label")
plt.tight_layout()
plt.show()

### 15.2 ROC Curve (Cross-Validated)

In [ ]:
# ROC curve using cross-validated (held-out) probabilities


mask = ~np.isnan(oof_proba)
y_true = y.values[mask]
y_score = oof_proba[mask]

fpr, tpr, thresholds = roc_curve(y_true, y_score)
auc_val = roc_auc_score(y_true, y_score)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUC = {auc_val:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")  # random baseline
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC curve (cross-validated)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 15.3 ROC Curves (5-Fold CV)

In [ ]:
# 5 ROC curves (per fold) + mean ROC

mean_fpr = np.linspace(0, 1, 101)
interp_tprs = []

plt.figure(figsize=(6, 6))

for i, (fpr, tpr, a) in enumerate(zip(fprs, tprs, aucs), start=1):
    plt.plot(fpr, tpr, alpha=0.4, label=f"Fold {i} (AUC={a:.3f})")
    interp_tprs.append(np.interp(mean_fpr, fpr, tpr))

mean_tpr = np.mean(interp_tprs, axis=0)
mean_tpr[0] = 0.0
mean_tpr[-1] = 1.0
mean_auc = sk_auc(mean_fpr, mean_tpr)

plt.plot(mean_fpr, mean_tpr, linewidth=2, label=f"Mean ROC (AUC={mean_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")  # random baseline

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC curves (5-fold CV)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 15.4 Precision–Recall Curve

In [ ]:
# Precision–Recall curve (often what people mean by "recall curve")


mask = ~np.isnan(oof_proba)
y_true = y.values[mask]
y_score = oof_proba[mask]

precision, recall, thresholds = precision_recall_curve(y_true, y_score)
ap = average_precision_score(y_true, y_score)

plt.figure(figsize=(6, 6))
plt.plot(recall, precision, label=f"AP = {ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curve (cross-validated)")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

### 15.5 Confusion Matrix (threshold = 0.5)

In [ ]:
# Confusion matrix at threshold = 0.5 (using cross-validated probabilities)


mask = ~np.isnan(oof_proba)
y_true = y.values[mask]
y_pred = (oof_proba[mask] >= 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["0", "1"])

plt.figure(figsize=(5, 5))
disp.plot(values_format="d")
plt.title("Confusion matrix (threshold = 0.5)")
plt.tight_layout()
plt.show()

## 16. Liver Pathology Categories — Most Common Raw Strings

In [ ]:
# Liver pathology categories: top most common raw strings
liver_path[cat_col].fillna("NA").value_counts()

## 17. Impute Missing Liver Pathology Categories from Notes

Strategy: keyword/regex matching — build a synonym dictionary from known
categories, validate on ground-truth rows, then apply to the missing rows.
Uses **ConText-inspired negation detection** to avoid false positives.

### 17.1 Category Patterns Dictionary

In [ ]:
CATEGORY_PATTERNS = {
    # ── core / liver ─────────────────────────────────────────────────────────
    "steatosis":          r"steatosis|steatotic|macrovesicular\s+steat|microvesicular\s+steat",
    "congestion":         r"congest|congnest|sinusoidal dilat",
    "fibrosis":           r"fibros|fibrous|fibrotic|bridging",
    "cirrhosis":          r"cirrh",
    "inflammation":       r"inflam|lymphocyte|lymphoid\s+infiltrat|infiltrat",
    "necrosis":           r"necrosi|necrotic",
    "hepatitis":          r"hepatitis",
    "atrophy":            r"atrophy|atrophic",
    "hemorrhage":         r"h[ae]?morrhag",
    "hyalinization":      r"hyaliniz",
    "infarction":         r"infarct",
    "nodularity":         r"nodular|nodule",
    "hyperplasia":        r"hyperplasia|hyperplastic",
    "sclerotic":          r"sclerotic|sclerosis",
    "pigment":            r"pigment",
    "no_abnormalities":   r"no abnormal|no\s+major\s+abnormal|within normal|unremarkable"
                          r"|no patholog|no\s+active",
    "clean_specimens":    r"\bclean\b|no lesion|good specimens?|excellent specimens?"
                          r"|well preserved|good preservation|good morphology"
                          r"|excellent preservation|good mucosal preservation",
    # ── overrides ────────────────────────────────────────────────────────────
    "ischemic_changes":   r"ischemi",
    "scarring":           r"\bscar\b|scarr",
    "corpora_albicantia": r"corpora\s+albi[ck]an",
    "post_menopausal":    r"post[- ]?menopausal|post\s+menopausal",
    "sweat_glands":       r"sweat\s+glands?",
    "solar_elastosis":    r"solar\s+elastosis",
    "heart_failure_cells":r"heart\s+failure\s+cells?",
    "calcification":      r"calc[iy]f",
    "atherosclerosis":    r"atheroscler",
    "glomerulosclerosis": r"glomerulosclerosis|glomerular\s+sclerosis"
                          r"|sclerosed?\s+glomerul|sclerotic\s+glomerul",
    "goiter":             r"goit",
    "esophagitis":        r"esophagitis|reflux",
    "hypereosinophilia":  r"hypereosinophil",
    "atelectasis":        r"at[ae]lectasis",
    "diabetic":           r"diabet",
}


### 17.2 ConText Negation Detection & Extraction Function

In [ ]:
_CLAUSE_SPLIT = re.compile(r"[.;]")

_SCOPE_TERM_SPLIT = re.compile(
    r"\b(?:and|but|however|yet|although|except|presenting|presents)\b"
    r"|consistent\s+with|\bc/w\b",
    re.IGNORECASE,
)

_POSITIVE_QUALIFIER = re.compile(
    r"^\s*(mild|moderate|severe|marked|significant|diffuse|focal|minimal|prominent"
    r"|central|passive|active|chronic|acute|slight|extensive|occasional|few|several"
    r"|some|scattered|moderately|focally|diffusely|markedly|mildly|slightly"
    r"|predominantly|unusual|notable|rare|incidental)",
    re.IGNORECASE,
)

_NEGATION_TRIGGER = re.compile(
    r"\b(no|not|without|absent|absence\s+of|negative\s+for|free\s+of"
    r"|no\s+evidence\s+of|no\s+sign\s+of|denies|denied|rules?\s+out|ruled\s+out)\b",
    re.IGNORECASE,
)

_NORMAL_CATS = {"no_abnormalities", "clean_specimens"}


def _smart_comma_split(text):
    parts = text.split(",")
    if len(parts) == 1:
        return parts
    merged = [parts[0]]
    for part in parts[1:]:
        if _POSITIVE_QUALIFIER.match(part):
            merged.append(part)
        else:
            merged[-1] += "," + part
    return merged


def _is_negated_in_subclause(sub, match_start):
    return bool(_NEGATION_TRIGGER.search(sub[:match_start]))


def extract_categories(note_text):
    if not isinstance(note_text, str) or note_text.strip() == "":
        return []
    matched = set()
    for clause in _CLAUSE_SPLIT.split(note_text):
        for scope_chunk in _SCOPE_TERM_SPLIT.split(clause):
            for sub in _smart_comma_split(scope_chunk):
                sub = sub.strip()
                if not sub:
                    continue
                for cat, rx in COMPILED.items():
                    if cat in matched:
                        continue
                    m = rx.search(sub)
                    if m and not _is_negated_in_subclause(sub, m.start()):
                        matched.add(cat)
    if matched - _NORMAL_CATS:
        matched -= _NORMAL_CATS
    return sorted(matched)


COMPILED = {cat: re.compile(pat, re.IGNORECASE) for cat, pat in CATEGORY_PATTERNS.items()}


### 17.3 Sanity Checks

In [ ]:
test_cases = [
    ("2 pieces; mild macrovesicular steatosis",                    {"steatosis"}),
    ("2 pieces; no steatosis, mild congestion",                    {"congestion"}),
    ("no evidence of fibrosis; no congestion",                     set()),
    ("congestion present; without steatosis",                      {"congestion"}),
    ("steatosis; absent inflammation; mild fibrosis",              {"fibrosis", "steatosis"}),
    ("2 pieces; congestion and steatosis",                         {"congestion", "steatosis"}),
    ("no steatosis but congestion noted",                          {"congestion"}),
    ("no steatosis and no congestion",                             set()),
    ("no steatosis and congestion",                                {"congestion"}),
    ("2 pieces; no fibrosis, inflammation or fat",                 set()),
    ("2 pieces; no infiltrates, fibrosis, congestion or fat",      set()),
    ("2 pieces, 8x7 & 6x6mm; no fat, fibrosis or excess congestion", set()),
    ("2 pieces; minimal fat",                                      set()),
    ("2 pieces; thick fibrous trabeculae",                         {"fibrosis"}),
    ("few collections of lymphocytes, often in portal tracts",     {"inflammation"}),
    ("2 pieces; atypical lymphoid infiltrates",                    {"inflammation"}),
    ("central sinusoidal dilatation",                              {"congestion"}),
    ("giant cell granulomas",                                      set()),
    ("2 pieces; no fat or inflammation",                           set()),
    ("2 pieces; <2% macrovesicular fat; no fibrosis or inflammation", set()),
    ("2 pieces; congestion and mild fatty change",                 {"congestion"}),
    ("Appears cirrhotic, request trichrome",                       {"cirrhosis"}),
    ("2 pieces, moderate chronic passive congnestion",             {"congestion"}),
    ("severe fibrosis without nodule formation consistent with cardiac cirrhosis",
                                                                   {"fibrosis", "cirrhosis"}),
    ("2 pieces; <5% macrovesicular steatosis; no active or chronic lesions",
                                                                   {"steatosis"}),
    ("2 pieces; good preservation, no lesions",                    {"clean_specimens"}),
    ("2 pieces; good morphology",                                  {"clean_specimens"}),
    ("2 pieces; Mallory hyaline bodies, bridging fibrosis",        {"fibrosis"}),
    ("2 pieces; hemorrhagic infarction",                           {"hemorrhage", "infarction"}),
    ("Centrilobular ischemia and congestion",                      {"congestion", "ischemic_changes"}),
    ("2 pieces; focal hemorrhages; capsule sampled",               {"hemorrhage"}),
    ("unusual rim of adherent hmorrhage/clot",                     {"hemorrhage"}),
    ("6 pieces; good representation of lymphoid tissue",           set()),
    ("medial calcific sclerosis",                                   {"calcification", "sclerotic"}),
    ("4 pieces; atherosclerotic changes with calcification",       {"atherosclerosis", "calcification", "sclerotic"}),
    ("2 pieces; hyalinized tubules, no spermatogenesis",           {"hyalinization"}),
    ("2 pieces; nodular goitre",                                   {"goiter", "nodularity"}),
    ("6 pieces; squamous hyperplasia consistent with reflux",      {"esophagitis", "hyperplasia"}),
    ("2 pieces; atalectasis",                                      {"atelectasis"}),
    ("2 pieces; severe (diabetic) glomerulosclerosis",             {"diabetic", "glomerulosclerosis", "sclerotic"}),
    ("2 pieces no abnormalities, unusual rim of adherent hmorrhage/clot",
                                                                   {"hemorrhage"}),
]

print("Sanity checks (v4):")
all_pass = True
for note, expected in test_cases:
    got = set(extract_categories(note))
    status = "PASS" if got == expected else "FAIL"
    if status == "FAIL":
        all_pass = False
        print(f"  [{status}] {note!r}")
        print(f"         Expected: {expected}")
        print(f"         Got     : {got}")
print("All sanity checks passed!" if all_pass else "Some checks FAILED.")


### 17.4 Extend Vocabulary from All-Tissue Data

In [ ]:
_n_hardcoded = len(CATEGORY_PATTERNS)
_observed = (
    df_meta_url["Pathology.Categories"]
    .dropna().str.split(r",\s*").explode().str.strip().str.lower().unique()
)
for _cat in _observed:
    if _cat and _cat not in CATEGORY_PATTERNS:
        CATEGORY_PATTERNS[_cat] = re.escape(_cat)

COMPILED = {cat: re.compile(pat, re.IGNORECASE) for cat, pat in CATEGORY_PATTERNS.items()}
print(f"Vocab: {_n_hardcoded} hardcoded + {len(CATEGORY_PATTERNS)-_n_hardcoded} from data"
      f" = {len(CATEGORY_PATTERNS)} total")


### 17.5 Validate on Ground-Truth Rows

Apply patterns to notes where categories are already known and measure
per-category **precision, recall, specificity, F1** (TP / FP / FN / TN).

In [ ]:
liver_all = df_meta_url[df_meta_url["Tissue"].astype(str) == "Liver"].copy()
liver_gt = liver_all[liver_all["Pathology.Categories"].notna()].copy()
print(f"Total liver rows: {len(liver_all)}")
print(f"Ground-truth rows (categories present): {len(liver_gt)}")

def parse_gt_categories(raw: str) -> set:
    return {c.strip().lower() for c in str(raw).split(",") if c.strip()}

liver_gt["gt_set"]   = liver_gt["Pathology.Categories"].apply(parse_gt_categories)
liver_gt["pred_set"] = liver_gt["Pathology.Notes"].apply(lambda n: set(extract_categories(n)))

rows = []
n = len(liver_gt)
for cat in sorted(CATEGORY_PATTERNS.keys()):
    in_gt   = liver_gt["gt_set"].apply(lambda s: cat in s)
    in_pred = liver_gt["pred_set"].apply(lambda s: cat in s)
    tp = int(( in_gt &  in_pred).sum())
    fp = int((~in_gt &  in_pred).sum())
    fn = int(( in_gt & ~in_pred).sum())
    tn = int((~in_gt & ~in_pred).sum())
    precision   = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall      = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    f1          = 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else float("nan")
    rows.append({"category": cat, "support": tp+fn, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                 "precision": round(precision, 3), "recall": round(recall, 3),
                 "specificity": round(specificity, 3), "F1": round(f1, 3)})

val_df = pd.DataFrame(rows).sort_values("support", ascending=False)
display(val_df)

covered = set(CATEGORY_PATTERNS.keys())
total_gt   = liver_gt["gt_set"].apply(lambda s: len(s & covered)).sum()
total_pred = liver_gt.apply(lambda r: len(r["gt_set"] & r["pred_set"] & covered), axis=1).sum()
total_fp   = liver_gt.apply(lambda r: len(r["pred_set"] - r["gt_set"]), axis=1).sum()
print(f"\nOverall recall  (within vocab): {total_pred}/{total_gt} = {total_pred/total_gt:.3f}")
print(f"Total false positives across all rows: {total_fp}")


### 17.6 FP / FN Case Review

In [ ]:
review_cols = ["Pathology.Categories", "Pathology.Notes"]
error_cats = val_df[(val_df["FP"] > 0) | (val_df["FN"] > 0)]["category"].tolist()
print(f"{len(error_cats)} categories with errors: {error_cats}\n")

for cat in error_cats:
    in_gt   = liver_gt["gt_set"].apply(lambda s: cat in s)
    in_pred = liver_gt["pred_set"].apply(lambda s: cat in s)
    fp_rows = liver_gt[ ~in_gt &  in_pred][review_cols].copy()
    fn_rows = liver_gt[  in_gt & ~in_pred][review_cols].copy()
    if fp_rows.empty and fn_rows.empty:
        continue
    sep = "=" * 70
    print(f"{sep}")
    print(f"  CATEGORY: {cat.upper()}   |  FP={len(fp_rows)}  FN={len(fn_rows)}")
    print(f"{sep}")
    if not fp_rows.empty:
        print(f"\n  -- FALSE POSITIVES (predicted '{cat}' but GT does not have it) --")
        display(fp_rows.reset_index(drop=True))
    if not fn_rows.empty:
        print(f"\n  -- FALSE NEGATIVES (GT has '{cat}' but regex missed it) --")
        display(fn_rows.reset_index(drop=True))


### 17.7 Apply Regex to Missing Rows

In [ ]:
liver_missing = liver_all[liver_all["Pathology.Categories"].isna()].copy()
print(f"Rows with missing categories: {len(liver_missing)}")

liver_missing["pred_list"] = liver_missing["Pathology.Notes"].apply(extract_categories)
liver_missing["Pathology.Categories.Imputed"] = liver_missing["pred_list"].apply(
    lambda lst: ", ".join(lst) if lst else float("nan")
)

filled   = liver_missing["Pathology.Categories.Imputed"].notna().sum()
unfilled = liver_missing["Pathology.Categories.Imputed"].isna().sum()
print(f"Rows newly assigned categories : {filled} / {len(liver_missing)} ({100*filled/len(liver_missing):.1f}%)")
print(f"Rows still unclassified (no match): {unfilled}")


### 17.8 Audit & Save Imputed Labels

In [ ]:
sample_cols = ["Tissue.Sample.ID", "Pathology.Notes", "Pathology.Categories.Imputed"]
print("=== Random sample of all newly imputed rows ===")
display(
    liver_missing[liver_missing["Pathology.Categories.Imputed"].notna()]
    [sample_cols]
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

if unfilled > 0:
    print(f"\n=== {unfilled} rows where no pattern matched ===")
    display(
        liver_missing[liver_missing["Pathology.Categories.Imputed"].isna()]
        [["Tissue.Sample.ID", "Pathology.Notes"]]
        .reset_index(drop=True)
    )

liver_full = liver_all.copy()
liver_full["Pathology.Categories.Imputed"] = liver_full["Pathology.Categories"]
liver_full.loc[liver_missing.index, "Pathology.Categories.Imputed"] = (
    liver_missing["Pathology.Categories.Imputed"]
)
liver_full["Pathology.Categories.Final"] = liver_full["Pathology.Categories"].fillna(
    liver_full["Pathology.Categories.Imputed"]
)

out_path = processed_dir / "liver_pathology_labels_imputed.csv"
liver_full.to_csv(out_path, index=False)
print(f"\nSaved {len(liver_full)} rows to {out_path}")
print(f"\nFinal category coverage:")
print(f"  Originally filled : {liver_full['Pathology.Categories'].notna().sum()}")
print(f"  After imputation  : {liver_full['Pathology.Categories.Final'].notna().sum()}")
print(f"  Still missing     : {liver_full['Pathology.Categories.Final'].isna().sum()}")


## 18. Multi-Category Liver Pathology Models (Blood-Based)

For every liver pathology category with **≥ positive samples** (including steatosis):

A. **Donor labels** from `Pathology.Categories.Final` → **Yes / No / NA**  
B. **Map to Whole Blood** samples (attach each donor's label)  
C. **Leak-free 5-fold grouped CV** — AUC-based top-100 feature selection inside each fold  
D. **Plots**: ROC-AUC, Precision–Recall AUC, and confusion matrix using
   Youden's J optimal threshold (maximises sensitivity + specificity)

In [ ]:
# ── Multi-category liver pathology models ──────────────────────────────────────
# Selects ALL liver categories with >= SAMPLE_THRESHOLD positive samples.

from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score, mean_absolute_error, auc as sk_auc,
    confusion_matrix, ConfusionMatrixDisplay,
)

SAMPLE_THRESHOLD = 5  # minimum number of positive donors required to model a category

# ── 1. Load imputed liver labels ──────────────────────────────────────────────
liver_imp = pd.read_csv(processed_dir / "liver_pathology_labels_imputed.csv")

# Discover eligible categories (including steatosis)
cat_counts = Counter()
for val in liver_imp["Pathology.Categories.Final"].dropna():
    for c in val.split(", "):
        cat_counts[c.strip()] += 1

eligible_cats = sorted(c for c, n in cat_counts.items() if n >= SAMPLE_THRESHOLD)
print(f"Eligible categories (≥{SAMPLE_THRESHOLD} samples): {eligible_cats}\n")

# ── 2. Donor-level labels: Yes / No / NA ─────────────────────────────────────
liver_known = liver_imp[liver_imp["Pathology.Categories.Final"].notna()].copy()
all_donors = liver_imp["SUBJID"].unique()

donor_labels = {}   # {category: Series(SUBJID → 0/1)}
label_tables = {}   # {category: Series(SUBJID → Yes/No/NA)}

for cat in eligible_cats:
    has_cat = liver_known["Pathology.Categories.Final"].str.contains(cat, case=False).astype(int)
    donor_lab = has_cat.groupby(liver_known["SUBJID"]).max()
    donor_labels[cat] = donor_lab

    label_series = pd.Series("NA", index=all_donors, name=cat)
    label_series[donor_lab.index] = donor_lab.map({1: "Yes", 0: "No"}).values
    label_tables[cat] = label_series

    pos = int(donor_lab.sum())
    neg = int((donor_lab == 0).sum())
    na_ = int(label_series.eq("NA").sum())
    print(f"  {cat:20s}  Yes={pos:4d}  No={neg:4d}  NA={na_:4d}")

# ── 3. Export combined label table ────────────────────────────────────────────
label_df_all = pd.DataFrame(label_tables)
label_df_all.index.name = "SUBJID"
label_df_all.to_csv(tables_dir / "labels_liver_multicategory.csv")
print(f"\nSaved donor-level labels → {tables_dir / 'labels_liver_multicategory.csv'}")

# ── 4. Map to blood samples & run leak-free 5-fold CV per category ───────────
# blood_subjid was defined in the steatosis section above
model_results = {}

for cat in eligible_cats:
    print(f"\n{'='*60}")
    print(f"  Category: {cat}")
    print(f"{'='*60}")

    y_cat = blood_subjid.map(donor_labels[cat])
    keep = y_cat.notna()
    X_cat = X_wb.loc[keep].copy()
    y_cat = y_cat.loc[keep].astype(int)
    g_cat = blood_subjid.loc[keep].astype(str)

    print(f"  Blood samples: {len(y_cat)}  (pos={int(y_cat.sum())}, neg={int((y_cat==0).sum())})")

    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(solver="saga", max_iter=5000)),
    ])

    oof = np.full(len(y_cat), np.nan)
    fold_fprs, fold_tprs, fold_aucs = [], [], []

    for fold, (tr, te) in enumerate(cv.split(X_cat, y_cat, groups=g_cat), 1):
        Xtr, Xte = X_cat.iloc[tr], X_cat.iloc[te]
        ytr, yte = y_cat.iloc[tr], y_cat.iloc[te]

        # Feature selection (AUC-based, top 100)
        strengths = {}
        for col in Xtr.columns:
            s = pd.to_numeric(Xtr[col], errors="coerce").fillna(0)
            if s.nunique() < 2:
                strengths[col] = 0.0
                continue
            a = roc_auc_score(ytr, s)
            strengths[col] = abs(a - 0.5)
        scores = pd.Series(strengths).reindex(Xtr.columns).fillna(0)
        top_feat = scores.nlargest(100).index.tolist()

        pipe.fit(Xtr[top_feat], ytr)
        proba = pipe.predict_proba(Xte[top_feat])[:, 1]
        oof[te] = proba

        fauc = roc_auc_score(yte, proba)
        fpr, tpr, _ = roc_curve(yte, proba)
        fold_fprs.append(fpr)
        fold_tprs.append(tpr)
        fold_aucs.append(fauc)
        print(f"  Fold {fold}: AUC = {fauc:.3f}")

    mean_auc = np.mean(fold_aucs)
    std_auc = np.std(fold_aucs)
    print(f"  → Mean AUC = {mean_auc:.3f} ± {std_auc:.3f}")

    # ── Youden's J optimal threshold (maximises sensitivity + specificity) ────
    mask_oof = ~np.isnan(oof)
    fpr_oof, tpr_oof, thresholds_oof = roc_curve(y_cat.values[mask_oof], oof[mask_oof])
    j_scores = tpr_oof - fpr_oof
    best_idx = np.argmax(j_scores)
    optimal_thresh = thresholds_oof[best_idx]
    print(f"  → Youden's J optimal threshold = {optimal_thresh:.3f}")

    model_results[cat] = {
        "y": y_cat, "oof": oof,
        "fold_fprs": fold_fprs, "fold_tprs": fold_tprs, "fold_aucs": fold_aucs,
        "mean_auc": mean_auc, "std_auc": std_auc,
        "optimal_threshold": optimal_thresh,
    }

# Save summary
summary_rows = [
    {"category": c, "mean_auc": r["mean_auc"], "std_auc": r["std_auc"],
     "optimal_threshold": r["optimal_threshold"]}
    for c, r in model_results.items()
]
pd.DataFrame(summary_rows).to_csv(tables_dir / "cv_results_liver_multicategory.csv", index=False)
print(f"\nSaved CV summary → {tables_dir / 'cv_results_liver_multicategory.csv'}")

### 18.1 AUC-ROC Curves (Per Category, 5-Fold CV + Mean)

In [ ]:
# ── AUC-ROC: per-fold curves + mean, one subplot per category (2-row layout) ─
n_cats = len(eligible_cats)
ncols = math.ceil(n_cats / 2)
nrows = 2 if n_cats > 1 else 1

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)

for idx, cat in enumerate(eligible_cats):
    r, c = divmod(idx, ncols)
    ax = axes[r, c]
    res = model_results[cat]
    mean_fpr = np.linspace(0, 1, 101)
    interp_tprs = []

    for i, (fpr, tpr, a) in enumerate(
        zip(res["fold_fprs"], res["fold_tprs"], res["fold_aucs"]), 1
    ):
        ax.plot(fpr, tpr, alpha=0.35, label=f"Fold {i} ({a:.3f})")
        interp_tprs.append(np.interp(mean_fpr, fpr, tpr))

    mean_tpr = np.mean(interp_tprs, axis=0)
    mean_tpr[0], mean_tpr[-1] = 0.0, 1.0
    mean_auc_val = sk_auc(mean_fpr, mean_tpr)

    ax.plot(mean_fpr, mean_tpr, lw=2,
            label=f"Mean ({mean_auc_val:.3f})")
    ax.plot([0, 1], [0, 1], ls="--", color="grey")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.set_title(f"{cat}\nAUC = {res['mean_auc']:.3f} ± {res['std_auc']:.3f}")
    ax.legend(fontsize=7, loc="lower right")

# Hide unused subplots
for idx in range(n_cats, nrows * ncols):
    r, c = divmod(idx, ncols)
    axes[r, c].set_visible(False)

fig.suptitle("ROC curves — liver pathology (blood-based, 5-fold CV)", y=1.02)
fig.tight_layout()
fig.savefig(figures_dir / "roc_liver_multicategory.pdf", bbox_inches="tight")
fig.savefig(figures_dir / "roc_liver_multicategory.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {figures_dir / 'roc_liver_multicategory.pdf'}")

### 18.2 Precision–Recall Curves (Per Category, Cross-Validated)

In [ ]:
# ── Precision-Recall: one subplot per category (2-row layout) ─────────────────

ncols = math.ceil(n_cats / 2)
nrows = 2 if n_cats > 1 else 1

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)

pr_summary = []

for idx, cat in enumerate(eligible_cats):
    r, c = divmod(idx, ncols)
    ax = axes[r, c]
    res = model_results[cat]
    y_true = res["y"].values
    y_score = res["oof"]
    mask = ~np.isnan(y_score)

    precision, recall, _ = precision_recall_curve(y_true[mask], y_score[mask])
    ap = average_precision_score(y_true[mask], y_score[mask])
    prevalence = y_true[mask].mean()

    ax.plot(recall, precision, lw=2, label=f"AP = {ap:.3f}")
    ax.axhline(prevalence, ls="--", color="grey", label=f"Baseline = {prevalence:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"{cat}\nAP = {ap:.3f}")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7, loc="upper right")

    pr_summary.append({"category": cat, "avg_precision": ap, "prevalence": prevalence})

# Hide unused subplots
for idx in range(n_cats, nrows * ncols):
    r, c = divmod(idx, ncols)
    axes[r, c].set_visible(False)

fig.suptitle("Precision–Recall curves — liver pathology (blood-based, CV)", y=1.02)
fig.tight_layout()
fig.savefig(figures_dir / "pr_liver_multicategory.pdf", bbox_inches="tight")
fig.savefig(figures_dir / "pr_liver_multicategory.png", dpi=150, bbox_inches="tight")
plt.show()

pr_df = pd.DataFrame(pr_summary)
print("\nPrecision-Recall summary:")
display(pr_df)
print(f"\nSaved → {figures_dir / 'pr_liver_multicategory.pdf'}")

### 18.3 Confusion Matrices (Youden's J Optimal Threshold)

In [ ]:
# ── Confusion matrices: Youden's J optimal threshold (2-row layout) ───────────

ncols = math.ceil(n_cats / 2)
nrows = 2 if n_cats > 1 else 1

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False)

for idx, cat in enumerate(eligible_cats):
    r, c = divmod(idx, ncols)
    ax = axes[r, c]
    res = model_results[cat]
    y_true = res["y"].values
    y_score = res["oof"]
    mask = ~np.isnan(y_score)
    thresh = res["optimal_threshold"]

    y_pred = (y_score[mask] >= thresh).astype(int)
    cm = confusion_matrix(y_true[mask], y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No", "Yes"])
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"{cat}\n(threshold = {thresh:.3f}, Youden's J)")

# Hide unused subplots
for idx in range(n_cats, nrows * ncols):
    r, c = divmod(idx, ncols)
    axes[r, c].set_visible(False)

fig.suptitle("Confusion matrices — liver pathology (Youden's J threshold, CV)", y=1.02)
fig.tight_layout()
fig.savefig(figures_dir / "cm_liver_multicategory.pdf", bbox_inches="tight")
fig.savefig(figures_dir / "cm_liver_multicategory.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {figures_dir / 'cm_liver_multicategory.pdf'}")

### 18.4 Box Plots — Predicted Probability by Ground-Truth Label

In [ ]:
# ── Box plots: predicted probability by label (2-row layout) ──────────────────

ncols = math.ceil(n_cats / 2)
nrows = 2 if n_cats > 1 else 1

fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4 * nrows), squeeze=False)

for idx, cat in enumerate(eligible_cats):
    r, c = divmod(idx, ncols)
    ax = axes[r, c]
    res = model_results[cat]
    y_true = res["y"].values
    y_score = res["oof"]
    mask = ~np.isnan(y_score)
    thresh = res["optimal_threshold"]

    p0 = y_score[mask & (y_true == 0)]
    p1 = y_score[mask & (y_true == 1)]

    ax.boxplot([p0, p1], vert=False, tick_labels=["No", "Yes"])

    ax.set_xlim(0, 1)
    ax.set_xlabel("Predicted probability (class = Yes)")
    ax.set_ylabel("Ground truth")
    ax.set_title(f"{cat}")

# Hide unused subplots
for idx in range(n_cats, nrows * ncols):
    r, c = divmod(idx, ncols)
    axes[r, c].set_visible(False)

fig.suptitle("Cross-validated predicted probability by label — liver pathology", y=1.02)
fig.tight_layout()
fig.savefig(figures_dir / "boxplot_liver_multicategory.pdf", bbox_inches="tight")
fig.savefig(figures_dir / "boxplot_liver_multicategory.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {figures_dir / 'boxplot_liver_multicategory.pdf'}")